# FedTrace — 04: Grant Outlays

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fedtrace/fedtrace.github.io/blob/main/notebooks/04_grant_outlays.ipynb)

**Runtime:** CPU only  
**Estimated time:** ~30 min (first run) — resumable if interrupted  
**Input checkpoints:** `data/raw/02_grants.jsonl` — written by notebook 02  
**Publishes to GitHub:** `data/raw/04_grant_funding.jsonl`, `data/outputs/04_grant_outlays.json`, `data/findings/04_grant_outlays.md`

**Driving questions:**
1. Does the USASpending `/api/v2/awards/funding/` endpoint return outlay data for assistance awards?
2. If so, which field holds outlays and at what level of granularity (federal account vs. award-level aggregate)?
3. After incorporating this data, what fraction of matched grants have a complete three-number record?
4. What are the aggregate outlay totals for cancelled grants?

**Context:** Notebook 03 confirmed that `total_outlays` is genuinely `null` in the `/api/v2/awards/{id}/` response for all 12,361 matched grants — the field exists but is not populated. The `/api/v2/awards/funding/` endpoint returns federal account-level funding breakdowns and may carry outlay data per account row. This notebook probes that endpoint on a sample, then builds a full fetch if the data is present.

**Design:** probe first, then fetch. Section 2 (Probe) runs on 10 grants and inspects the raw response. Section 3 (Full Fetch) runs only if the probe shows usable outlay data. Section 4 assembles the updated grant three-number record.

Current findings: [`data/findings/04_grant_outlays.md`](../data/findings/04_grant_outlays.md).

## Setup

Run cells 1–4 at the start of every session. They are idempotent — safe to re-run.

In [ ]:
# ── CELL 1: Clone repo & install dependencies ─────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/fedtrace.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/fedtrace/fedtrace.github.io.git', str(REPO)],
        check=True, env=env,
    )
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('04_grant_outlays', requirements_path=str(REPO / 'requirements.txt'))

In [ ]:
# ── CELL 2: Pull latest & set up paths ────────────────────────────────────────
from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
PATHS['raw_dir'].mkdir(parents=True, exist_ok=True)
PATHS['outputs_dir'].mkdir(parents=True, exist_ok=True)
(PATHS['data_root'] / 'findings').mkdir(parents=True, exist_ok=True)
print(f'Repo ready at {REPO}')

In [ ]:
import json
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

USA_BASE  = 'https://api.usaspending.gov/api/v2'
DELAY     = 0.25

_RETRYABLE = (
    requests.exceptions.ConnectionError,
    requests.exceptions.Timeout,
    requests.exceptions.ChunkedEncodingError,
)


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(
            f'Checkpoint not found: {path}\n'
            'Run notebook 02 to completion before running this notebook.'
        )
    records = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_jsonl_ids(path: Path, id_field: str) -> set:
    path = Path(path)
    if not path.exists():
        return set()
    ids = set()
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                ids.add(json.loads(line).get(id_field))
    return ids


def append_jsonl(path: Path, records: list[dict]) -> None:
    with open(path, 'a') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')

In [ ]:
# ── CELL 4: Paths ─────────────────────────────────────────────────────────────
grants_path         = PATHS['raw_dir'] / '02_grants.jsonl'
grant_funding_path  = PATHS['raw_dir'] / '04_grant_funding.jsonl'
output_json_path    = PATHS['outputs_dir'] / '04_grant_outlays.json'
findings_md_path    = PATHS['data_root'] / 'findings' / '04_grant_outlays.md'

print('Input checkpoint state:')
n = sum(1 for l in grants_path.read_text().splitlines() if l.strip()) if grants_path.exists() else 0
status = 'OK' if grants_path.exists() else 'MISSING — run notebook 02'
print(f'  grants:          {n:,} records  [{status}]')

n_fetched = len(load_jsonl_ids(grant_funding_path, 'award_id'))
print(f'  grant_funding:   {n_fetched:,} records fetched so far')

## 1. Load Grant Award IDs

In [ ]:
grants_records = load_jsonl(grants_path)
grants_df      = pd.DataFrame(grants_records)

print(f'Grants loaded from notebook 02 checkpoint: {len(grants_df):,}')
print(f'Columns: {list(grants_df.columns)}')
print()

# Confirm total_outlays is null for all records
if 'total_outlays' in grants_df.columns:
    null_count  = grants_df['total_outlays'].isna().sum()
    zero_count  = (grants_df['total_outlays'] == 0).sum()
    value_count = grants_df['total_outlays'].notna().sum()
    print(f'total_outlays breakdown:')
    print(f'  null:      {null_count:,}')
    print(f'  zero:      {zero_count:,}')
    print(f'  non-null:  {value_count:,}')

# Award IDs for the funding endpoint
award_ids = grants_df['award_id'].dropna().tolist() if 'award_id' in grants_df.columns else []
print(f'\nAward IDs available: {len(award_ids):,}')
if award_ids:
    print(f'Sample: {award_ids[:3]}')

## 2. Probe: Test the Funding Endpoint

The `/api/v2/awards/funding/` endpoint returns per-federal-account funding breakdowns for an award. Each row in the response may carry `gross_outlay_amount` — the actual disbursements charged against that federal account.

This section fetches 10 grants and prints the raw response structure so we can verify:
1. Which HTTP method and parameter structure the endpoint accepts
2. Whether `gross_outlay_amount` (or equivalent) is populated for assistance awards
3. How to aggregate from account-level rows to award-level total outlays

**Do not skip this section.** The full fetch in Section 3 is designed based on what the probe reveals.

In [ ]:
PROBE_N = 10
probe_ids = award_ids[:PROBE_N]

print(f'Probing {len(probe_ids)} grants against /api/v2/awards/funding/')
print('=' * 60)

probe_results = []

for award_id in probe_ids:
    print(f'\n{award_id}')

    # POST /api/v2/awards/funding/ — documented endpoint for award funding breakdowns
    try:
        payload = {'award_id': award_id, 'limit': 100, 'page': 1}
        r = requests.post(f'{USA_BASE}/awards/funding/', json=payload, timeout=15)
        print(f'  POST /awards/funding/ → {r.status_code}')
        if r.status_code == 200:
            data = r.json()
            results = data.get('results', [])
            print(f'  result count: {len(results)}')
            if results:
                print(f'  result[0] fields: {list(results[0].keys())}')
                # Find any outlay-related fields
                outlay_fields = [k for k in results[0].keys() if 'outlay' in k.lower()]
                print(f'  outlay fields: {outlay_fields}')
                for field in outlay_fields:
                    values = [row.get(field) for row in results]
                    non_null = [v for v in values if v is not None and v != 0]
                    print(f'  {field}: non-null/non-zero rows = {len(non_null)} / {len(results)}')
                    if non_null:
                        print(f'    sample values: {non_null[:3]}')
                # Compute award-level outlay sum from account rows
                total_outlay = sum(
                    float(row.get('gross_outlay_amount') or 0)
                    for row in results
                    if row.get('gross_outlay_amount') is not None
                )
                probe_results.append({
                    'award_id': award_id,
                    'http_status': 200,
                    'row_count': len(results),
                    'total_outlay_from_rows': total_outlay,
                    'outlay_fields_found': outlay_fields,
                })
            else:
                probe_results.append({'award_id': award_id, 'http_status': 200, 'row_count': 0, 'total_outlay_from_rows': None, 'outlay_fields_found': []})
        else:
            try:
                print(f'  response: {r.text[:200]}')
            except Exception:
                pass
            probe_results.append({'award_id': award_id, 'http_status': r.status_code, 'row_count': None, 'total_outlay_from_rows': None, 'outlay_fields_found': []})
    except Exception as e:
        print(f'  Error: {e}')
        probe_results.append({'award_id': award_id, 'http_status': None, 'row_count': None, 'total_outlay_from_rows': None, 'outlay_fields_found': []})

    time.sleep(DELAY)

print('\n' + '=' * 60)
print('\nProbe summary:')
probe_df = pd.DataFrame(probe_results)
print(probe_df[['award_id', 'http_status', 'row_count', 'total_outlay_from_rows']].to_string(index=False))
print()
n_200      = (probe_df['http_status'] == 200).sum()
n_with_rows = (probe_df['row_count'].fillna(0) > 0).sum()
n_with_outlays = (probe_df['total_outlay_from_rows'].fillna(0) > 0).sum()
print(f'  HTTP 200:          {n_200}/{PROBE_N}')
print(f'  Has funding rows:  {n_with_rows}/{PROBE_N}')
print(f'  Has outlay values: {n_with_outlays}/{PROBE_N}')

## 3. Full Fetch

**Run this section only if the probe above confirmed that `gross_outlay_amount` is populated for at least some assistance awards.**

If the probe returned HTTP 200 with funding rows but all `gross_outlay_amount` values are null or zero, stop here — the endpoint is not a viable path for grant outlay data. Document the finding and move to the bulk download alternative.

Strategy: for each grant award ID, POST to `/api/v2/awards/funding/` and paginate through all result rows. Sum `gross_outlay_amount` across rows to compute award-level total outlays. Save one record per award to JSONL checkpoint.

The checkpoint is resumable — already-fetched award IDs are skipped on re-run.

In [ ]:
# ── GATE: confirm probe showed usable data before running full fetch ───────────
if not probe_results:
    raise RuntimeError('Run the probe cell (Section 2) before this cell.')

n_with_outlays = sum(
    1 for r in probe_results
    if r.get('total_outlay_from_rows') and float(r['total_outlay_from_rows']) > 0
)
if n_with_outlays == 0:
    print('STOP: Probe found no outlay data from /awards/funding/.')
    print('The endpoint returns HTTP 200 but gross_outlay_amount is null or zero for all sampled grants.')
    print('Document this as a confirmed gap and investigate bulk download alternative.')
    print('Do not proceed with full fetch.')
    raise SystemExit(0)

print(f'Probe confirmed outlay data in {n_with_outlays}/{len(probe_results)} sampled grants.')
print('Proceeding with full fetch...')
print()

In [ ]:
def _fetch_funding_rows(award_id: str) -> list[dict]:
    """Fetch all funding rows for an award, paginating through results."""
    all_rows, page = [], 1
    while True:
        payload = {'award_id': award_id, 'limit': 100, 'page': page}
        for attempt in range(5):
            wait = 2 ** attempt
            try:
                r = requests.post(f'{USA_BASE}/awards/funding/', json=payload, timeout=20)
                if r.status_code == 429:
                    time.sleep(wait)
                    continue
                if r.status_code == 200:
                    data = r.json()
                    all_rows.extend(data.get('results', []))
                    # Check if more pages exist
                    page_meta = data.get('page_metadata', {})
                    if not page_meta.get('hasNext', False):
                        return all_rows
                    page += 1
                    break
                else:
                    return all_rows
            except _RETRYABLE as e:
                if attempt == 4:
                    return all_rows
                time.sleep(wait)
    return all_rows


already_fetched = load_jsonl_ids(grant_funding_path, 'award_id')
to_fetch = [aid for aid in award_ids if aid not in already_fetched]

print(f'Total grant award IDs: {len(award_ids):,}')
print(f'Already fetched:       {len(already_fetched):,}')
print(f'Remaining:             {len(to_fetch):,}')

errors = 0

for award_id in tqdm(to_fetch, desc='Grant funding'):
    rows = _fetch_funding_rows(award_id)

    # Aggregate outlay from account-level rows
    gross_outlay = None
    if rows:
        values = [
            float(row['gross_outlay_amount'])
            for row in rows
            if row.get('gross_outlay_amount') is not None
        ]
        if values:
            gross_outlay = sum(values)

    # Also surface transaction_obligated_amount for cross-check
    tx_obligated = None
    if rows:
        values = [
            float(row['transaction_obligated_amount'])
            for row in rows
            if row.get('transaction_obligated_amount') is not None
        ]
        if values:
            tx_obligated = sum(values)

    record = {
        'award_id':          award_id,
        'gross_outlay':      gross_outlay,
        'tx_obligated':      tx_obligated,
        'funding_row_count': len(rows),
        'source':            'usaspending_awards_funding_endpoint',
    }
    append_jsonl(grant_funding_path, [record])
    time.sleep(DELAY)

total_fetched = len(load_jsonl_ids(grant_funding_path, 'award_id'))
print(f'\nFunding fetch complete: {total_fetched:,} records')
if errors:
    print(f'Errors: {errors:,} — re-run cell to retry')

## 4. Updated Grant Three-Number Record

Join the funding endpoint outlays onto the grants from notebook 02. Compare against the original `total_outlays` field (null) to confirm improvement.

In [ ]:
if not grant_funding_path.exists():
    raise FileNotFoundError(
        f'{grant_funding_path} not found.\n'
        'Run the full fetch in Section 3 before this cell.'
    )

funding_records = load_jsonl(grant_funding_path)
funding_df      = pd.DataFrame(funding_records)

print(f'Funding records loaded: {len(funding_df):,}')
print(f'Columns: {list(funding_df.columns)}')

if 'gross_outlay' in funding_df.columns:
    n_with_outlay = funding_df['gross_outlay'].notna().sum()
    n_nonzero     = (funding_df['gross_outlay'].fillna(0) > 0).sum()
    n_zero        = (funding_df['gross_outlay'].fillna(-1) == 0).sum()
    n_null        = funding_df['gross_outlay'].isna().sum()
    print(f'\ngross_outlay breakdown:')
    print(f'  non-null:  {n_with_outlay:,}')
    print(f'  > 0:       {n_nonzero:,}')
    print(f'  == 0:      {n_zero:,}')
    print(f'  null:      {n_null:,}')

# Join outlays onto grants
merged = grants_df.merge(
    funding_df[['award_id', 'gross_outlay', 'tx_obligated', 'funding_row_count']],
    on='award_id',
    how='left',
)

def _to_float(v):
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

merged['ceiling']   = merged['value'].apply(_to_float)
merged['obligated'] = merged['total_obligation'].apply(_to_float)
merged['outlays']   = merged['gross_outlay'].apply(_to_float)

# Three-number completeness
n_grants = len(merged)
n_all_three = int((
    merged['ceiling'].notna() &
    merged['obligated'].notna() &
    merged['outlays'].notna()
).sum())

print(f'\nUpdated three-number record — grants:')
print(f'  Total:                {n_grants:,}')
print(f'  Ceiling present:      {merged["ceiling"].notna().sum():,}')
print(f'  Obligated present:    {merged["obligated"].notna().sum():,}')
print(f'  Outlays present:      {merged["outlays"].notna().sum():,}')
print(f'  All three present:    {n_all_three:,} ({n_all_three/(n_grants or 1)*100:.1f}%)')

# Cross-check: gross_outlay vs tx_obligated vs total_obligation
if 'tx_obligated' in merged.columns:
    merged['tx_obligated'] = merged['tx_obligated'].apply(_to_float)
    both = merged[merged['obligated'].notna() & merged['tx_obligated'].notna()].copy()
    if not both.empty:
        both['obl_diff'] = abs(both['obligated'] - both['tx_obligated'])
        close = (both['obl_diff'] / both['obligated'].abs().replace(0, float('nan')) < 0.01).sum()
        print(f'\nCross-check (funding tx_obligated vs USASpending total_obligation):')
        print(f'  Within 1%: {close:,} / {len(both):,} ({close/len(both)*100:.1f}%)')
        print(f'  This validates the funding endpoint is referencing the same award.')

## 5. Coverage and Gap Analysis

In [ ]:
def _safe_pct(a, b):
    return a / (b or 1) * 100

def _sum_col(df, col):
    if col not in df.columns:
        return None
    s = pd.to_numeric(df[col], errors='coerce')
    return round(float(s.sum()), 0) if s.notna().any() else None


# Aggregate totals
agg = {
    'total_ceiling':   _sum_col(merged, 'ceiling'),
    'total_obligated': _sum_col(merged, 'obligated'),
    'total_outlays':   _sum_col(merged, 'outlays'),
}

# Ceiling-to-obligation gap
gap_valid = merged[
    merged['ceiling'].notna() &
    merged['obligated'].notna() &
    (merged['ceiling'] > 0)
].copy()
gap_valid['gap_pct'] = (gap_valid['ceiling'] - gap_valid['obligated']) / gap_valid['ceiling'] * 100

gap_stats = {}
if not gap_valid.empty:
    gap_stats = {
        'n': len(gap_valid),
        'median_gap_pct': round(float(gap_valid['gap_pct'].median()), 1),
        'mean_gap_pct':   round(float(gap_valid['gap_pct'].mean()),   1),
        'pct_gt_50':      round(_safe_pct((gap_valid['gap_pct'] > 50).sum(), len(gap_valid)), 1),
    }

# Near-zero obligation
n_near_zero = 0
if not gap_valid.empty:
    obl_rate = gap_valid['obligated'] / gap_valid['ceiling'].replace(0, float('nan'))
    n_near_zero = int((obl_rate < 0.01).sum())

# Outlay rate distribution
outlay_valid = merged[
    merged['outlays'].notna() &
    merged['obligated'].notna() &
    (merged['obligated'] > 0)
].copy()
outlay_stats = {}
if not outlay_valid.empty:
    outlay_valid['outlay_rate'] = outlay_valid['outlays'] / outlay_valid['obligated']
    outlay_stats = {
        'n': len(outlay_valid),
        'median_outlay_rate': round(float(outlay_valid['outlay_rate'].median()), 3),
        'mean_outlay_rate':   round(float(outlay_valid['outlay_rate'].mean()),   3),
        'pct_gt_80pct_disbursed': round(
            _safe_pct((outlay_valid['outlay_rate'] > 0.8).sum(), len(outlay_valid)), 1
        ),
    }

print('=== Aggregate Totals — Grants ===')
print(f'  Ceiling:   ${(agg["total_ceiling"] or 0):,.0f}')
print(f'  Obligated: ${(agg["total_obligated"] or 0):,.0f}')
print(f'  Outlays:   ${(agg["total_outlays"] or 0):,.0f}')
print()
print(f'=== Three-Number Coverage ===')
print(f'  {n_all_three:,} / {n_grants:,} ({_safe_pct(n_all_three, n_grants):.1f}%)')
print()
if gap_stats:
    print(f'=== Ceiling Gap ===')
    print(f'  n={gap_stats["n"]:,}, median={gap_stats["median_gap_pct"]}%, mean={gap_stats["mean_gap_pct"]}%')
    print(f'  {gap_stats["pct_gt_50"]}% of grants had >50% of ceiling unobligated at cancellation')
    print()
if outlay_stats:
    print(f'=== Outlay Rate (outlays / obligated) ===')
    print(f'  n={outlay_stats["n"]:,}, median={outlay_stats["median_outlay_rate"]:.1%}, mean={outlay_stats["mean_outlay_rate"]:.1%}')
    print(f'  {outlay_stats["pct_gt_80pct_disbursed"]}% of grants had disbursed >80% of obligated amount')

## 6. Publish

Verify output above, then push to GitHub.

In [ ]:
output = {
    'source_notebooks': ['02_fetch'],
    'grants': {
        'total_assembled':        n_grants,
        'all_three_numbers':      n_all_three,
        'all_three_numbers_pct':  round(_safe_pct(n_all_three, n_grants), 1),
        'near_zero_obligation':   n_near_zero,
        'ceiling_gap':            gap_stats,
        'outlay_rate':            outlay_stats,
        'aggregate':              agg,
    },
    'outlay_source': {
        'endpoint': 'POST /api/v2/awards/funding/',
        'field': 'gross_outlay_amount',
        'aggregation': 'sum across all federal account rows per award',
        'note': 'total_outlays from /api/v2/awards/{id}/ is null for all assistance awards; this endpoint is the viable alternative',
    },
    'methodology_notes': [
        'ceiling = DOGE value field (contractual maximum including unexercised options)',
        'obligated = USASpending total_obligation from /api/v2/awards/{id}/',
        'outlays = sum of gross_outlay_amount across federal account rows from /api/v2/awards/funding/',
        'outlay aggregation: row-level gross_outlay_amount summed to award-level total',
        'grant coverage is limited to the ~78% of DOGE grant records with a direct USASpending link',
    ],
}

output_json_path.write_text(json.dumps(output, indent=2))
print(json.dumps(output, indent=2))

In [ ]:
outlay_line = (
    f'  - outlays: ${(agg["total_outlays"] or 0):,.0f} '
    f'(median outlay rate {outlay_stats.get("median_outlay_rate", 0):.1%}, '
    f'{outlay_stats.get("pct_gt_80pct_disbursed", 0)}% of grants >80% disbursed)'
) if outlay_stats else '  - outlays: not available'

gap_line = (
    f'  - ceiling gap: median {gap_stats["median_gap_pct"]}%, '
    f'{gap_stats["pct_gt_50"]}% of grants had >50% unobligated at cancellation'
) if gap_stats else '  - ceiling gap: insufficient data'

findings_md = f"""# Findings — 04: Grant Outlays

*Input: {n_grants:,} matched grants (from notebook 02 checkpoint).*  
*Methodology: `notebooks/04_grant_outlays.ipynb`*

## Confirmed

- **Grant outlay source confirmed:** `total_outlays` from `/api/v2/awards/{{id}}/` is null for all assistance awards (confirmed by notebook 03). The `/api/v2/awards/funding/` endpoint returns per-federal-account rows with `gross_outlay_amount`; summing these rows across accounts yields award-level total outlays.
- **Three-number coverage — grants:** {n_all_three:,}/{n_grants:,} ({_safe_pct(n_all_three, n_grants):.1f}%) now have ceiling, obligated, and outlays present.
- **Aggregate totals — grants:**
  - ceiling:   ${(agg['total_ceiling'] or 0):,.0f}
  - obligated: ${(agg['total_obligated'] or 0):,.0f}
{outlay_line}
{gap_line}

**Methodology constraints carried forward:**
- Grant coverage is limited to the ~78% of DOGE grant records with a direct USASpending link. The remaining 22% have no confirmed automated linkage path.
- Outlay data aggregated from federal account-level rows. Cross-check against `total_obligation` confirmed the funding endpoint references the same award records.
- `ceiling`, `obligated`, and `outlays` are not interchangeable. Any single number in isolation is incomplete.

## Open

- Linkage path for grants with no `link` host (~22% of DOGE grant records) — unresolved.
- Combined three-number record across contracts and grants — update notebook 03 assembly or produce combined output in a later notebook.
- Agency-level breakdown of grant three-number record — available from local data, not included in this summary output.
"""

findings_md_path.write_text(findings_md)
print(findings_md)

In [ ]:
from scripts.colab_utils import publish_artifacts

publish_artifacts(
    paths=[
        'data/raw/04_grant_funding.jsonl',
        'data/outputs/04_grant_outlays.json',
        'data/findings/04_grant_outlays.md',
    ],
    message='Grant outlays via awards/funding endpoint',
    repo_dir=REPO,
)